# Tree ensembles vs linear models (Prompt #10, Steps 19-22, 44-47, 81-82)

Random forest, XGBoost, LightGBM and CatBoost on the tree-comparison pairs, LightGBM on every primary pair. Complexity must earn its place: a tree family is preferred only when it beats the reference linear model block by block by more than one SE. The randomized searches count as trials (ML-H ledger); a winner whose one-step neighbours collapse is not a stable winner.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.ml` and `xauusd_quant.research.ml_*` and read back from `results/ml_research/`. Generate those first:

```
xq ml-research --timeframe 5m [--stages grid,trees,...]   # cached walk-forward units
xq ml-report --timeframe 5m                              # tables, ML-H ledger rows, plots
xq ml-compare --timeframe 5m --target mean_reversion      # one target, from the units
xq ml-freeze --timeframe 5m                              # pre-registered freeze rules
xq ml-finalize --timeframe 5m                            # artifacts, streaming, latency
xq ml-final-test --model-spec LGBM_REVERSION_5M_H5_V001   # once, frozen specs only
```

**Prediction research only.** Models output probabilities and expected values; there is no BUY / SELL, threshold, position, stop, PnL or execution anywhere. Walk-forward blocks 2011-2021 are development; the reserved period (2022-) is read only by `ml-final-test`, after the specs are frozen. Skill is measured against the constant training mean (log-loss / MSE skill, positive = better than the baseline) and read block by block.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.ml.config import load_ml_config

cfg = load_ml_config(ROOT / "config" / "ml.yaml")

TIMEFRAME = "5m"       # 5m | 15m - change and re-run

out = cfg.results_path / TIMEFRAME
if not (out / "tables").exists():
    print(f"No report for {TIMEFRAME}. Run:  xq ml-research --timeframe {TIMEFRAME} --report")

def table(name):
    """Load a previously generated table (None if it was not written)."""
    path = out / "tables" / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(*parts):
    path = out.joinpath(*parts)
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Trees vs the reference linear model

In [ ]:
table('paired_vs_linear').sort('target', 'horizon', 't', descending=[False, False, True])

## Mean skill by family

In [ ]:
summ = table('configuration_summary')
base = summ.filter(pl.col('variant') == 'base')
base.pivot(on='family', index=['target', 'horizon'], values='mean')

## Fit time and size

In [ ]:
base.select('target', 'horizon', 'family', 'fit_seconds').sort('fit_seconds', descending=True).head(20)

## Randomized search, neighbourhood, complexity

In [ ]:
plot('hyperparameter_stability')

In [ ]:
display(table('search'))
display(table('neighbourhood'))
table('complexity')

## Booster missing-value handling and class weights

In [ ]:
table('variants')

## Prediction correlation and disagreement across families

In [ ]:
plot('prediction_correlation')
plot('model_disagreement')
table('disagreement')

## Relationships between targets (LightGBM scores)

In [ ]:
table('multitask_correlation')